<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/12_dose_analysis/dvh_and_dose_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dose-Volume Histograms (DVH) and dose metrics

## Introduction

A **DVH** summarizes the dose delivered to a structure. pyCERR's `cerr.dvh` computes the histogram and a full set of metrics (`Dx`, `Vx`, mean/max dose, `MOHx`/`MOCx`, `eud`). We build a demo plan (sample CT + a synthetic target/OAR + a synthetic dose), plot cumulative DVHs and tabulate metrics.

## Install pyCERR

In [ ]:
!pip install "pyCERR @ git+https://github.com/cerr/pyCERR.git"

## Set up a demo plan

In [ ]:
import numpy as np
import cerr.plan_container as pc
from cerr import datasets

# Real sample CT (downloaded & cached on first use)
dcmDir = datasets.fetch_sample_data('head_and_neck')
planC = pc.loadDcmDir(dcmDir)

# --- add two synthetic structures: a target (PTV) and an OAR (Parotid) ---
nR, nC, nS = (int(v) for v in planC.scan[0].getScanSize())
xV, yV, zV = planC.scan[0].getScanXYZVals()        # grid coords (cm)
Xc, Yc, Zc = np.meshgrid(xV, yV, zV, indexing='xy')
cx, cy, cz = xV[nC // 2], yV[nR // 2], zV[nS // 2]
distPTV2 = (Xc - cx) ** 2 + (Yc - cy) ** 2 + (Zc - cz) ** 2
ptvMask = distPTV2 < 3.0 ** 2                       # 3 cm sphere
oarMask = ((Xc - (cx + 4.0)) ** 2 + (Yc - cy) ** 2
           + (Zc - cz) ** 2) < 1.5 ** 2            # 1.5 cm sphere, offset
planC = pc.importStructureMask(ptvMask.astype(np.uint8), 0, 'PTV', planC)
planC = pc.importStructureMask(oarMask.astype(np.uint8), 0, 'Parotid', planC)

# --- add a synthetic dose: 70 Gy across the PTV with a Gaussian fall-off ---
PRESCRIPTION = 70.0
dist = np.sqrt(distPTV2)
dose3M = PRESCRIPTION * np.exp(-(np.clip(dist - 3.0, 0, None) ** 2)
                              / (2 * 2.5 ** 2))
dose3M[ptvMask] = PRESCRIPTION
planC = pc.importDoseArray(dose3M, xV, yV, zV, planC, 0,
                           {'fractionGroupID': 'Demo', 'units': 'GY'})
print('scans=%d  structures=%s  doses=%d'
      % (len(planC.scan), [s.structureName for s in planC.structure],
         len(planC.dose)))

## Compute and plot the DVH

`getDVH` returns per-voxel doses and volumes for a structure/dose pair; `doseHist` bins them. The cumulative DVH is the reverse cumulative sum (% volume receiving at least each dose).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cerr import dvh

plt.figure(figsize=(6, 4))
for strNum, label, color in [(0, 'PTV', 'tab:red'),
                             (1, 'Parotid', 'tab:blue')]:
    dosesV, volsV, _ = dvh.getDVH(strNum, 0, planC)
    binW = max(float(dosesV.max()) / 400.0, 1e-3)
    binsV, histV = dvh.doseHist(dosesV, volsV, binW)
    cumPct = 100.0 * np.flip(np.cumsum(np.flip(histV))) / np.sum(histV)
    plt.plot(binsV, cumPct, color=color, lw=2, label=label)
plt.xlabel('Dose (Gy)'); plt.ylabel('Volume (%)'); plt.ylim(0, 105)
plt.grid(alpha=0.3); plt.legend(); plt.title('Cumulative DVH')
plt.show()

## Dose metrics

Common metrics from the binned DVH. For `Vx`/`Dx`, `volumeType=1` expresses volume as a fraction (x100 for %); `Dx`'s cutoff is then a percentage of the structure volume.

In [ ]:
rows = []
for strNum, name in [(0, 'PTV'), (1, 'Parotid')]:
    dosesV, volsV, _ = dvh.getDVH(strNum, 0, planC)
    binsV, histV = dvh.doseHist(dosesV, volsV,
                                max(float(dosesV.max()) / 400.0, 1e-3))
    rows.append((name,
                 dvh.meanDose(binsV, histV),
                 dvh.maxDose(binsV, histV),
                 dvh.Dx(binsV, histV, 95, 1),          # D95 (Gy)
                 dvh.Vx(binsV, histV, 60, 1) * 100,    # V60Gy (%)
                 dvh.eud(binsV, histV, 1.0)))          # EUD (a=1)
print('%-10s %8s %8s %8s %8s %8s'
      % ('struct', 'mean', 'max', 'D95', 'V60Gy%', 'EUD'))
for r in rows:
    print('%-10s %8.1f %8.1f %8.1f %8.1f %8.1f' % r)

## Notes

* Other metrics: `MOHx`/`MOCx` (mean of hottest/coldest x%), `minDose`, `medianDose`.
* To drive outcome models (NTCP/TCP) from these dose statistics, see `10_outcomes_modeling/normal_tissue_complication_probability.ipynb`.